In [ ]:
def media(age_range):
    nums = [int(s) for s in age_range.replace('-', ' ').split() if s.isdigit()]

    if len(nums) == 2:
        return sum(nums) / 2
    elif len(nums) == 1:
        return float(nums[0])
    return np.nan

df['AgeCategory'] = df['AgeCategory'].apply(media)

print(df['AgeCategory'].unique())

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, uniform
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

In [ ]:
unique_values, counts = np.unique(y, return_counts=True)
proportions = counts / len(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [ ]:
numerical_atributes = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_atributes = X_train.select_dtypes(include=['object']).columns.tolist()

In [ ]:
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()

In [ ]:
preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, numerical_atributes),
                                               ('cat', categorical_transformer, categorical_atributes)],
                                 remainder='passthrough'
                                )

In [ ]:
preprocessor

In [ ]:
pipe = Pipeline([('preprocessor', preprocessor),
                 ('classifier', SVC())])

In [ ]:
param_distributions = {
    'classifier__kernel': ['linear', 'poly', 'rbf'],
    'classifier__C': loguniform(0.01, 100),
    'classifier__degree': [2, 3, 4, 5],
    'classifier__coef0': uniform(0, 10),
    'classifier__gamma': ['scale', 'auto'] + list(loguniform(0.001, 1).rvs(5))
}

In [ ]:
random_search = RandomizedSearchCV(estimator=pipe,
                                   param_distributions=param_distributions,
                                   n_iter=10,
                                   cv=5,
                                   scoring='accuracy',
                                   random_state=42,
                                   n_jobs=-1,
                                   verbose=2)

In [ ]:
random_search.fit(X_train, y_train)

print(f"hiperparâmetros: {random_search.best_params_}")
print(f"acurácia: {random_search.best_score_}")

In [ ]:
y_pred = random_search.predict(X_test)

In [ ]:
matriz = confusion_matrix(y_test, y_pred)

display = ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=le.classes_)
display.plot()
plt.show()

In [ ]:
classification_report(y_test, y_pred)
print(classification_report(y_test, y_pred))